# 표준화와 정규화 실습

**Standardization · Normalization · Scaling**

변수의 단위와 범위를 맞추어 학습을 안정화하는 전처리.

소재 분야에서 이해하기: 온도(℃)와 시간(h)의 크기 차이를 없애고 학습한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 전처리 문서](https://scikit-learn.org/stable/modules/preprocessing.html)

## 1. 단위가 다르면 무슨 일이 생기나

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

print('변수별 범위:')
for name, column in zip(FEATURES, X.T):
    print('  %-8s %.1f ~ %.1f' % (name, column.min(), column.max()))

for name, model in [('k-최근접이웃', KNeighborsRegressor(5)), ('Ridge', Ridge(alpha=1.0)),
                    ('랜덤 포레스트', RandomForestRegressor(n_estimators=200, random_state=0))]:
    raw = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
    standardised = cross_val_score(make_pipeline(StandardScaler(), model), X, y, cv=5, scoring='r2').mean()
    print('%-14s 원본 R2 %.3f / 표준화 R2 %.3f' % (name, raw, standardised))

## 2. 스케일러 종류와 이상치

In [ ]:
X_outlier = X.copy()
X_outlier[:5, 0] = 5000.0        # 잘못 기록된 온도 5건

for name, scaler in [('StandardScaler', StandardScaler()), ('MinMaxScaler', MinMaxScaler()),
                     ('RobustScaler', RobustScaler())]:
    transformed = scaler.fit_transform(X_outlier)[:, 0]
    normal = transformed[5:]
    print('%-16s 정상값 범위 %.3f ~ %.3f' % (name, normal.min(), normal.max()))
print('\nMinMax는 이상치 하나에 전체 범위가 눌립니다. RobustScaler는 중위수와 사분위로 스케일해 덜 민감합니다.')
print('트리 기반 모델은 분할 기준만 쓰므로 스케일링이 거의 필요하지 않습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#scaling)을 여세요.